# NB-02 · Ingest & Chunk

**Tujuan:** Mengubah dokumen nyata (PDF) menjadi potongan teks (*chunk*) yang siap di-*retrieve* oleh sistem RAG.

## Kenapa *chunking* penting?

- **Konteks LLM terbatas.** Model bahasa memiliki batas panjang konteks (misalnya 4 096–128 000 token). Kita tidak bisa memasukkan seluruh dokumen ke dalam satu *prompt*.
- **Retrieval bekerja per-potongan.** Saat pengguna mengajukan pertanyaan, sistem mencari potongan yang paling relevan — bukan seluruh dokumen. Potongan yang terlalu panjang menurunkan presisi; potongan yang terlalu pendek kehilangan konteks.
- **Strategi berbeda, hasil berbeda.** Tidak ada satu strategi *chunking* yang selalu terbaik. Notebook ini membandingkan tiga pendekatan dan mengukur kualitasnya secara kuantitatif.

In [ ]:
# Install ingestion dependencies
!pip install -q docling pdfplumber sentence-transformers "transformers<5"

In [ ]:
# --- Navasena course bootstrap (Colab) ---
import os, sys
REPO = "navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/chmdznr/navasena-gen-ml-course.git
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))
from tools.rag_utils import TokenCounter, TextChunker, chunk_quality_score, split_sentences

## Memuat Dokumen

Kita menggunakan dua pendekatan untuk mengekstrak teks dari PDF:

| Pendekatan | Kelebihan | Kekurangan |
|---|---|---|
| **Docling** (utama) | Memahami *layout* dokumen (tabel, heading, kolom), menghasilkan Markdown terstruktur | Perlu waktu *download* model pertama kali |
| **pdfplumber** (fallback) | Ringan, cepat, sudah tersedia di Colab gratis | Hanya ekstraksi teks polos, tidak memahami struktur |

Sel di bawah mencoba memuat file PDF yang diunggah pengguna; jika tidak ada, menggunakan dokumen contoh yang sudah disertakan di repo.

In [ ]:
# Upload PDF (Colab UI) or fall back to the baked-in sample document
try:
    from google.colab import files
    up = files.upload()
    pdf_path = next(iter(up)) if up else None
except Exception:
    pdf_path = None

if not pdf_path:
    pdf_path = "navasena-gen-ml-course/05_rag/data/sample_id_document.pdf"  # baked-in fallback


def extract_text(path):
    """Extract text from a PDF using Docling (layout-aware) with pdfplumber as fallback."""
    try:
        from docling.document_converter import DocumentConverter
        return DocumentConverter().convert(path).document.export_to_markdown()
    except Exception as e:
        print(f"Docling gagal ({type(e).__name__}: {e}); pakai pdfplumber.")
        import pdfplumber
        with pdfplumber.open(path) as pdf:
            return "\n\n".join((pg.extract_text() or "") for pg in pdf.pages)


raw_text = extract_text(pdf_path)
print(raw_text[:400])

## Menghitung Token dengan Tokenizer Qwen

Ukuran *chunk* yang bermakna harus diukur dalam **token**, bukan karakter — karena LLM bekerja dengan token. Kita menggunakan tokenizer Qwen2.5-3B-Instruct agar ukuran *chunk* di sini konsisten dengan model *generator* yang dipakai di `nb01_rag_fundamentals.ipynb`.

`TokenCounter` adalah *wrapper* yang bisa menerima fungsi tokenisasi apa pun (*injectable*), sehingga mudah diganti jika kita berganti model.

In [ ]:
# Wire the real Qwen tokenizer into the injectable TokenCounter
from transformers import AutoTokenizer

qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
counter = TokenCounter(tokenize_fn=lambda t: len(qwen_tok.encode(t)))

# Quick sanity check
sample = raw_text[:200]
print(f"200 karakter pertama = {counter.count(sample)} token (Qwen tokenizer)")

## Tiga Strategi Chunking

### 1. `sentence` — Per-kalimat (greedy pack)
Menggabungkan kalimat-kalimat secara serakah sampai batas token tercapai. **Kelebihan:** batas *chunk* selalu di akhir kalimat (koheren). **Kekurangan:** ukuran *chunk* bervariasi; kalimat sangat panjang bisa melebihi budget.

### 2. `fixed` — Jendela kata tetap dengan overlap
Memotong teks menjadi jendela kata berukuran tetap (diukur dalam token) dengan tumpang-tindih (*overlap*) di antara *chunk* berurutan. **Kelebihan:** ukuran *chunk* sangat konsisten. **Kekurangan:** bisa memotong di tengah kalimat; frasa yang berada di batas bisa kehilangan konteks.

### 3. `semantic` — Per-paragraf
Memisahkan teks berdasarkan baris kosong (pemisah paragraf alami). Paragraf yang melebihi budget di-*sub-chunk* dengan strategi `sentence`. **Kelebihan:** mempertahankan unit semantik dokumen. **Kekurangan:** sangat bergantung pada kualitas format dokumen asli. Overlap tidak dipakai pada strategi semantic karena antar-paragraf biasanya tidak berbagi konteks yang perlu diulang.

> **Parameter kunci:** `overlap_words` (bukan `overlap_tokens`) — diukur dalam *word*, bukan subword token.

In [ ]:
# Build chunks with all three strategies and score each one
# Note: overlap_words is measured in words, not subword tokens
sent_chunker  = TextChunker(counter, max_tokens=128, overlap_words=0)
fixed_chunker = TextChunker(counter, max_tokens=128, overlap_words=24)

results = {
    "sentence": sent_chunker.sentence(raw_text),
    "fixed":    fixed_chunker.fixed(raw_text),
    "semantic": sent_chunker.semantic(raw_text),
}

print(f"{'Strategi':10s}  {'# chunk':>8s}  {'quality score':>14s}")
print("-" * 38)
for name, chunks in results.items():
    score = chunk_quality_score(chunks, max_tokens=128, original_len=len(raw_text))
    print(f"{name:10s}  {len(chunks):>8d}  {score:>14.1f}")

## Membaca Skor Kualitas

`chunk_quality_score` menghasilkan skor 0–100 yang menggabungkan tiga aspek:

1. **Budget adherence** — seberapa banyak *chunk* yang berada dalam batas `max_tokens`. Nilai 100 berarti semua *chunk* di bawah budget.
2. **Konsistensi ukuran** — *chunk* yang seragam memudahkan sistem retrieval memperkirakan biaya konteks.
3. **Coverage** — total token semua *chunk* harus mendekati panjang dokumen asli (tidak ada teks yang hilang).

Strategi `semantic` biasanya menang untuk dokumen terstruktur (laporan, artikel) karena paragraf adalah unit makna alami. Strategi `fixed` biasanya lebih baik untuk teks yang tidak terstruktur atau dokumen yang dikonversi dari OCR dengan baris pendek-pendek.

In [ ]:
# Comparison table + token-size histogram per strategy (CPU only — no GPU needed)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, chunks) in zip(axes, results.items()):
    ax.hist([c.n_tokens for c in chunks], bins=10)
    ax.axvline(128, color="red", linestyle="--", label="max_tokens")
    ax.set_title(f"{name} (n={len(chunks)})")
    ax.set_xlabel("token per chunk")
    ax.legend()
axes[0].set_ylabel("jumlah chunk")
plt.tight_layout()
plt.show()

## Memilih Chunk Terbaik dan Meng-*embed*

Setelah membandingkan tiga strategi, kita memilih hasil terbaik untuk dokumen ini dan mengubah setiap *chunk* menjadi vektor *embedding*. Vektor ini yang nantinya disimpan di *vector store* dan digunakan oleh `nb03` untuk retrieval.

Kita menggunakan model **`paraphrase-multilingual-MiniLM-L12-v2`** — model yang sama dengan `nb01` — agar vektor yang dihasilkan kompatibel di seluruh pipeline RAG.

In [ ]:
# Embed the chosen strategy's chunks with the multilingual embedder
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Pilih strategi dengan skor tertinggi (biarkan kode yang memutuskan, bukan hardcode)
best_name = max(results, key=lambda n: chunk_quality_score(
    results[n], max_tokens=128, original_len=len(raw_text)))
print(f"Strategi terpilih: {best_name}")
best_chunks = results[best_name]
chunk_texts = [c.text for c in best_chunks]
chunk_vectors = embedder.encode(chunk_texts, convert_to_numpy=True)

print(f"{len(chunk_texts)} chunk -> embedding shape {chunk_vectors.shape}")
# chunk_texts + chunk_vectors menjadi input untuk nb03 (retrieve + rerank)

## Ringkasan dan Jembatan ke NB-03

Di notebook ini kita telah:

1. **Memuat dokumen** dari PDF menggunakan Docling (layout-aware) dengan fallback ke pdfplumber.
2. **Mengukur token** dengan tokenizer Qwen2.5-3B-Instruct yang sama dengan model *generator*.
3. **Membandingkan tiga strategi** *chunking* (`sentence`, `fixed`, `semantic`) dan mengukur kualitasnya.
4. **Meng-*embed*** *chunk* terpilih menjadi vektor siap-retrieve.

### Selanjutnya: NB-03 — Retrieve & Rerank

`chunk_texts` dan `chunk_vectors` yang dihasilkan di sini akan menjadi input untuk `nb03`. Di sana kita akan belajar:

- **Over-fetch** — mengambil lebih banyak kandidat dari yang dibutuhkan (mis. top-20).
- **Cross-encoder reranking** — menggunakan model *cross-encoder* untuk me-*rerank* kandidat berdasarkan relevansi semantik yang lebih dalam.
- **Mengapa dua tahap ini** jauh lebih baik daripada langsung mengambil top-k dari *vector store*.